In [ ]:
# Load shared paths from the repo-root config.py
import sys, os
sys.path.insert(0, os.path.abspath("../.."))   # repo root (run this notebook from its experiment folder)
import config


In [ ]:
from model import *
from evaluation_helper import evaluate_full_pipeline
import pandas as pd
test_data = pd.read_parquet(config.TECHNIQUE_TEST_PARQUET)

# IMPORTANT (same fix as training)
test_data["techniques"] = test_data["techniques"].apply(parse_labels)
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# rebuild label map (same as training)
label2id, id2label = build_label_map(test_data)

model = TechniqueClassifier(CFG.model_name, len(label2id))
checkpoint = torch.load("best_technique_model.pt")

model.load_state_dict(checkpoint["model_state_dict"], strict=False)

label2id = checkpoint["label2id"]
id2label = checkpoint["id2label"]
model.to(CFG.device)
evaluate_full_pipeline(
    model=model,
    test_df=test_data,
    dataset_class=TechniqueDataset,
    tokenizer=tokenizer,
    label2id=label2id,
    id2label=id2label,
    scorer_script_path="task-TC_scorer.py",
    techniques_list_path="propaganda-techniques-names-semeval2020task11.txt",
    output_dir="eval_results",
    device=CFG.device
)

In [ ]:
import torch
ckpt = torch.load("best_span_roberta_pos_ner_discourse_updated.pt")

print(ckpt["cfg"])